# Station 03｜GAN：莫內風格概念工作室

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/day6_ai_solution_lab/03_gan_monet_studio.ipynb)

**客戶任務：** 把旅遊照片轉成莫內風格概念稿，同時記錄內容變形、文字／人臉失真與處理時間。

- Kaggle 教學資料：[Monet2Photo](https://www.kaggle.com/datasets/balraj98/monet2photo)（非配對 Monet／Photo 影像）
- 經典題目：[I’m Something of a Painter Myself](https://www.kaggle.com/competitions/gan-getting-started/data)
- **範圍聲明：** 30 分鐘輪轉不從零訓練 CycleGAN。本 Notebook 的可跑 Candidate 是快速 Neural Style Transfer 代理模型；完整專題才換成教師已驗證的 CycleGAN Generator checkpoint。


## 1. Setup 與資料來源


In [ ]:
%pip -q install kagglehub==1.0.2 gradio==6.20.0 tensorflow-hub==0.16.1


In [ ]:
from pathlib import Path
import time

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image, ImageFilter

SEED = 20260719
IMAGE_SIZE = (256, 256)
tf.random.set_seed(SEED)
np.random.seed(SEED)


In [ ]:
DATASET_HANDLE = "balraj98/monet2photo"
data_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
all_images = sorted(
    path for path in data_root.rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

def domain(path):
    relative_parts = [part.lower() for part in path.relative_to(data_root).parts[:-1]]
    if any(part in {"monet", "monet_jpg", "traina", "testa"} for part in relative_parts):
        return "monet"
    if any(part in {"photo", "photo_jpg", "trainb", "testb"} for part in relative_parts):
        return "photo"
    return "unknown"

monet_files = [path for path in all_images if domain(path) == "monet"]
photo_files = [path for path in all_images if domain(path) == "photo"]
if not monet_files or not photo_files:
    raise RuntimeError("無法辨認 Monet／Photo 資料夾，請把 data_root 內容交給講師。")

monet_path = monet_files[SEED % len(monet_files)]
photo_path = photo_files[SEED % len(photo_files)]
style_image = Image.open(monet_path).convert("RGB").resize(IMAGE_SIZE)
content_image = Image.open(photo_path).convert("RGB").resize(IMAGE_SIZE)

print("Data root:", data_root)
print("Monet/Photo files:", len(monet_files), len(photo_files))
display(style_image, content_image)


## 2. Baseline：規則式色彩轉換

這不是 GAN；它提供一個低成本起點，幫助我們判斷 Candidate 是否真的增加價值。


In [ ]:
def color_transfer_baseline(content, style):
    content_array = np.asarray(content, dtype=np.float32) / 255
    style_array = np.asarray(style, dtype=np.float32) / 255
    c_mean = content_array.mean(axis=(0, 1), keepdims=True)
    c_std = content_array.std(axis=(0, 1), keepdims=True) + 1e-6
    s_mean = style_array.mean(axis=(0, 1), keepdims=True)
    s_std = style_array.std(axis=(0, 1), keepdims=True) + 1e-6
    transferred = (content_array - c_mean) / c_std * s_std + s_mean
    transferred = np.clip(transferred, 0, 1)
    return Image.fromarray((transferred * 255).astype(np.uint8))

baseline_image = color_transfer_baseline(content_image, style_image)
display(content_image, baseline_image)


## 3. Candidate：快速 Neural Style Transfer 代理模型

它是「預訓練影像轉換模型」的課堂代理，不可在報告中寫成 CycleGAN。若小組選 GAN 完整實作，需改用教師提供且有來源／雜湊的 Generator checkpoint。


In [ ]:
STYLE_MODEL_URL = "https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2"
style_model_available = True
try:
    style_model = hub.load(STYLE_MODEL_URL)
    content_tensor = tf.convert_to_tensor(np.asarray(content_image)[None, ...] / 255, dtype=tf.float32)
    style_tensor = tf.convert_to_tensor(np.asarray(style_image)[None, ...] / 255, dtype=tf.float32)
    started = time.perf_counter()
    stylized_tensor = style_model(content_tensor, style_tensor)[0]
    inference_seconds = time.perf_counter() - started
    stylized_array = np.clip(stylized_tensor[0].numpy(), 0, 1)
    candidate_raw = Image.fromarray((stylized_array * 255).astype(np.uint8))
except Exception as exc:
    style_model_available = False
    candidate_raw = baseline_image.filter(ImageFilter.DETAIL)
    inference_seconds = float("nan")
    print("代理模型暫時不可用，保留 Baseline 完成 blend 實驗：", type(exc).__name__, exc)


## 4. 必做實驗：改風格混合強度

`BLEND_ALPHA` 越高，代理模型輸出占比越高；請比較內容保留、風格感與破圖情形。


In [ ]:
BLEND_ALPHA = 0.75  # TODO(學員必改)：比較 0.40、0.75、1.00
candidate_image = Image.blend(content_image, candidate_raw.resize(IMAGE_SIZE), BLEND_ALPHA)

content_array = np.asarray(content_image, dtype=np.float32) / 255
output_array = np.asarray(candidate_image, dtype=np.float32) / 255
style_array = np.asarray(style_image, dtype=np.float32) / 255
content_ssim = float(tf.image.ssim(content_array, output_array, max_val=1.0).numpy())
palette_distance = float(np.linalg.norm(output_array.mean((0, 1)) - style_array.mean((0, 1))))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image, title in zip(
    axes,
    [content_image, style_image, candidate_image],
    ["Input photo", "Monet reference", f"Output (alpha={BLEND_ALPHA})"],
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
print({
    "content_ssim_higher_is_more_similar": round(content_ssim, 3),
    "palette_distance_lower_is_closer": round(palette_distance, 3),
    "inference_seconds": None if np.isnan(inference_seconds) else round(inference_seconds, 3),
    "proxy_model_available": style_model_available,
})


## 5. 完整專題的 CycleGAN checkpoint 介面（選案後再做）

教師需先驗證 checkpoint 的來源、授權、架構、SHA-256 與輸入前處理。沒有 checkpoint 時不要假裝已執行 CycleGAN。


In [ ]:
TEACHER_GENERATOR_PATH = Path("/content/drive/MyDrive/day6_assets/monet_generator.keras")

def load_teacher_generator(path=TEACHER_GENERATOR_PATH):
    if not path.exists():
        print("Optional checkpoint not found; rotation mode continues with the proxy model.")
        return None
    generator = tf.keras.models.load_model(path, compile=False)
    print("Loaded teacher-verified CycleGAN generator:", path.name)
    return generator

teacher_generator = load_teacher_generator()


## 6. 課堂暫時 Demo（選用）


In [ ]:
import gradio as gr

MODEL_VERSION = "station03-nst-proxy-v1" if style_model_available else "station03-color-baseline-v1"

def stylize_upload(image, alpha):
    if image is None:
        return None, {"status": "請先上傳圖片", "model_version": MODEL_VERSION}
    content = image.convert("RGB").resize(IMAGE_SIZE)
    if style_model_available:
        content_tensor = tf.convert_to_tensor(np.asarray(content)[None, ...] / 255, dtype=tf.float32)
        result = style_model(content_tensor, style_tensor)[0][0].numpy()
        raw = Image.fromarray((np.clip(result, 0, 1) * 255).astype(np.uint8))
    else:
        raw = color_transfer_baseline(content, style_image)
    output = Image.blend(content, raw.resize(IMAGE_SIZE), float(alpha))
    return output, {
        "model_version": MODEL_VERSION,
        "warning": "概念草稿；文字、人臉與細節可能失真。代理模型不是 CycleGAN。",
    }

demo = gr.Interface(
    fn=stylize_upload,
    inputs=[gr.Image(type="pil", label="上傳旅遊照片"), gr.Slider(0, 1, value=BLEND_ALPHA, label="風格強度")],
    outputs=[gr.Image(type="pil", label="風格結果"), gr.JSON(label="模型資訊")],
    title="莫內風格概念工作室（課堂暫時 Demo）",
)
print("需要介面時再執行：demo.launch(share=True)")


## 7. 下載迷你實驗卡


In [ ]:
experiment_card = f'''# GAN 站迷你實驗卡

- 組別：請填寫
- 我修改了：BLEND_ALPHA = {BLEND_ALPHA}
- 原本結果：請記錄修改前的內容保留／風格感
- 修改後結果：請記錄修改後的內容保留／風格感
- 內容 SSIM：{content_ssim:.3f}
- Palette distance：{palette_distance:.3f}
- 最大失敗情境：請填寫（人物、文字、夜景或其他）
- Seed：{SEED}
- 模型版本：{MODEL_VERSION}
- 限制：本站 Candidate 是快速 NST 代理模型，不是 CycleGAN。
'''
output_path = Path("/content/station03_gan_experiment_card.md")
output_path.write_text(experiment_card, encoding="utf-8")
print(experiment_card)
print("Saved:", output_path)
